# Geometric deep learning & equivariance — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Equivariant models transform their outputs predictably when the input geometry transforms
Geometric learning respects symmetries instead of relearning them. The examples verify rotation-invariant distances, rotation-equivariant linear maps, translation-invariant edge vectors, permutation equivariance, and an equivariant message update.

In [ ]:
# 1) distances are invariant to rotation
P=np.array([[1.,0.],[0.,1.]]); R=np.array([[0.,-1.],[1.,0.]]); Pr=P@R.T
d0=np.linalg.norm(P[0]-P[1]); d1=np.linalg.norm(Pr[0]-Pr[1])
plt.figure(figsize=(4,4)); plt.scatter(P[:,0],P[:,1],label='P'); plt.scatter(Pr[:,0],Pr[:,1],label='rotated'); plt.legend(); plt.axis('equal'); plt.title('distance invariant')
assert abs(d0-d1)<1e-9

In [ ]:
# 2) scalar multiplication commutes with rotation: equivariance
x=np.array([1.,2.]); R=np.array([[0.,-1.],[1.,0.]]); a=3.; left=R@(a*x); right=a*(R@x)
plt.figure(figsize=(5,3)); plt.bar(range(2),left-right); plt.title('R(a x) - a R(x) = 0')
assert np.allclose(left,right)

In [ ]:
# 3) edge vectors are translation-invariant
P=np.array([[1.,0.],[3.,1.]]); shift=np.array([5.,-2.]); e=P[1]-P[0]; e2=(P+shift)[1]-(P+shift)[0]
plt.figure(figsize=(4,4)); plt.arrow(P[0,0],P[0,1],e[0],e[1],head_width=.1,length_includes_head=True); plt.title('relative edge vector')
assert np.allclose(e,e2)

In [ ]:
# 4) graph message passing is permutation-equivariant
A=np.array([[0,1,0],[1,0,1],[0,1,0]],float); X=np.array([[1.],[2.],[4.]]); P=np.eye(3)[[2,0,1]]
left=(P@A@P.T)@(P@X); right=P@(A@X)
plt.figure(figsize=(5,3)); plt.bar(range(3),(left-right).ravel()); plt.title('permuted output difference')
assert np.allclose(left,right)

In [ ]:
# 5) radial message is rotation-equivariant
x=np.array([[1.,0.],[0.,1.]]); R=np.array([[0.,-1.],[1.,0.]]); msg=lambda Z: (Z[1]-Z[0])/(1+np.linalg.norm(Z[1]-Z[0]))
left=msg(x@R.T); right=msg(x)@R.T
plt.figure(figsize=(5,3)); plt.bar(range(2),left-right); plt.title('equivariant radial message')
assert np.allclose(left,right)